<a href="https://colab.research.google.com/github/Dineshseervi/AI_IIITM/blob/main/03_industrial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# W10 — Smart Home Agent: Industrial Implementation

**Level:** Industrial
**Stack:** LangChain v1 + LangGraph + Pydantic + Together AI
**Domain:** Smart Home (same as `02_patterns`)
**Time:** ~90 min

## Learning objectives

By the end of this notebook, you can:
- Take the **same 3-node BDI graph from `02_patterns`** and upgrade each node from hand-coded rules to an LLM prompt
- Migrate the BDI reasoning slots from `@dataclass` to `Pydantic BaseModel` with field validation
- Persist state across invocations with `MemorySaver` checkpointing
- Accumulate a `decision_log` audit trail across many BDI cycles
- Articulate what's still missing for true production

## Prerequisites

- `02_patterns.ipynb` — the rule-based version of this exact graph. We won't re-derive WHY there are three BDI nodes; we'll just plug an LLM into each of them.
- `_bridge_session.ipynb` — TypedDict vs dataclass vs Pydantic refresher. We'll use Pydantic for the BDI models here; the bridge explains why.

## Roadmap

1. **Section 0** — From patterns to industrial: what changes, what stays the same
2. **Section 1** — Pydantic models for the BDI slots (replacing `02_patterns`' dataclasses)
3. **Section 2** — Three prompts: one per BDI step (replacing `02_patterns`' rule blocks)
4. **Section 3** — Three nodes that each call an LLM (same shape as `02_patterns`)
5. **Section 4** — Build the graph and visualize (identical topology to `02_patterns`)
6. **Section 5** — Add a checkpointer for persistence
7. **Section 6** — Run several BDI cycles and watch reasoning accumulate
8. **Section 7** — Observability: read the audit trail
9. **Section 8** — What's still missing for true production


In [ ]:
!pip install langchain_together

In [ ]:
# Imports
import json
from operator import add                          # the reducer we'll use for decision_log
from typing import TypedDict, Optional, Literal, Annotated
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from IPython.display import Image, display

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Walk up from this notebook to find _global/.env
def _load_env():
    cwd = Path.cwd()
    for parent in [cwd] + list(cwd.parents):
        env_path = parent / ".env"
        if env_path.exists():
            load_dotenv(env_path)
            return env_path
    # fallback to current directory
    load_dotenv()
    return None

_env_loaded = _load_env()
print(f".env loaded from: {_env_loaded}")

PROVIDER = os.getenv("LLM_PROVIDER", "together").lower()

if PROVIDER == "together":
    from langchain_together import ChatTogether
    llm = ChatTogether(
        model=os.getenv("LLM_MODEL_DEFAULT", "Qwen/Qwen2.5-7B-Instruct-Turbo"),
        temperature=0.3,
        max_tokens=600,
    )
elif PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        model=os.getenv("LLM_MODEL_OPENAI", "gpt-4o-mini"),
        temperature=0.3,
    )
else:
    raise ValueError(f"Unknown LLM_PROVIDER: {PROVIDER}")

print(f"Provider: {PROVIDER}")


.env loaded from: None


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

## Setup check

If the cell above raised an error:
- Make sure `_global/.env` exists with `TOGETHER_API_KEY` set
- Run `python _global/scripts/verify_setup.py` from the repo root
- See `_global/README.md` for setup instructions


## Section 0 — From patterns to industrial

### 0.1 What we're KEEPING from `02_patterns`

This notebook is a strict extension of the BDI agent you built in `02_patterns.ipynb`. The architecture doesn't change. Specifically:

- **The graph topology is identical:** `START → update_belief → pick_desire → form_intention → END`. Three nodes in a chain. Same edges. Same names.
- **The BDI semantics are identical:** Belief is what the agent knows. Desire is the goal it picks. Intention is the action it commits to with a reason.
- **The Belief / Desire / Intention shapes are essentially identical:** same fields, just typed slightly more strictly with Pydantic.

If you understand `02_patterns.ipynb`, you understand 80% of this notebook already.

### 0.2 What we're CHANGING for production

| Layer | `02_patterns` | `03_industrial` | Why change |
|-------|---------------|-----------------|------------|
| BDI types | `@dataclass` | `Pydantic BaseModel` | Validation at boundaries; serializes to JSON for the checkpointer |
| `update_belief` logic | echoes input | **LLM prompt** parses raw observation into typed Belief | Agent observes the world from natural-language input, not from pre-structured dicts |
| `pick_desire` logic | if/elif on thresholds | **LLM prompt** picks goal given Belief | Reasoning becomes explainable; rules become natural-language guidelines |
| `form_intention` logic | if/elif on (goal, temp) | **LLM prompt** forms action+reason given Belief+Desire | Reasons are LLM-generated, more nuanced than fixed templates |
| State persistence | none (in-memory per invocation) | `MemorySaver` checkpointer | A home's BDI history survives across calls |
| Audit trail | not tracked | `decision_log` list of past intentions | A `/audit` endpoint can answer "what has this home decided over the last week?" |

The shape stays the same; the *substance* of each step becomes LLM-driven.

## Section 1 — Pydantic models for the BDI slots

If Pydantic feels new, see `_bridge_session.ipynb` Section 1.4. We'll only call out what's specific to migrating from dataclass.

### 1.1 Belief — same fields, now with validation

In [ ]:
class Belief(BaseModel):
    """What the agent knows about the world right now."""
    indoor_temp: float = Field(default=22.0, description="Celsius")
    outdoor_temp: float = Field(default=28.0, description="Celsius")
    occupancy: bool = True
    hour: int = Field(default=14, ge=0, le=23, description="Hour of day, 0-23")
    energy_price_now: float = Field(default=14.0, description="cents/kWh")

# Sanity check
b = Belief(indoor_temp=17.5, hour=8, energy_price_now=22.0)
print(b)
print(f"Pydantic model? {isinstance(b, BaseModel)}")

In [ ]:
b.model_dump()

### 1.2 Desire — `Literal` enforces the three valid goals

In `02_patterns` the `goal` field was a plain string. Bad values like `"goal": "comfortable"` slipped through. With Pydantic, `Literal["comfort", "save_energy", "safety"]` makes invalid goals impossible at construction time.

In [ ]:
class Desire(BaseModel):
    goal: Literal["comfort", "save_energy", "safety"] = "comfort"
    priority: int = Field(default=1, ge=1, le=3)


# Good
ok = Desire(goal="save_energy", priority=2)
print(f"OK: {ok}")

# Bad — Pydantic rejects
try:
    bad = Desire(goal="comfortable")  # typo would have slipped through dataclass
except Exception as e:
    print(f"REJECTED bad goal: {type(e).__name__}")

### 1.3 Intention — keeps the action and the reason

`action` stays a free-form string because the LLM may produce action names we haven't enumerated yet. `reason` is required and non-empty in practice — the whole point of BDI is to have an audit-trail-quality reason for every decision.

In [ ]:
class Intention(BaseModel):
    action: str = "STANDBY"
    reason: str = Field(default="", description="Human-readable justification")


print(Intention(action="HEAT_ON", reason="indoor temp 17.5C below comfort band"))

## Section 2 — Three prompts replace three rule blocks

Each BDI node in `02_patterns` was a rule block. Here, each node calls one LLM via one focused prompt. The prompts are the **only new thing students need to understand** beyond what they already know from patterns.

### 2.1 Belief prompt — parse a raw observation into structured Belief

In `02_patterns`, the user passed a pre-built `Belief(indoor_temp=17.5, ...)`. In production, the input arrives as **raw text** — a sensor report, a status string, a customer message. The belief node's job is to extract structured fields from that text.

In [ ]:
belief_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You extract structured beliefs from a raw smart-home observation.\n"
     "Return ONLY a JSON object with these exact keys (use sensible defaults for missing data):\n"
     '{{"indoor_temp": <float, C>, "outdoor_temp": <float, C>, "occupancy": <bool>, '
     '"hour": <int 0-23>, "energy_price_now": <float, cents/kWh>}}'),
    ("human", "Observation: {observation}")
])
belief_chain = belief_prompt | llm | JsonOutputParser()
print("belief_chain ready.")

### 2.2 Desire prompt — pick a goal given the Belief

In `02_patterns` this was an if/elif. Here it's a prompt with the same rules described in natural language. The LLM may produce more nuanced rationales than the hand-coded version (e.g., factoring in occupancy when temperature is moderate).

We use `.partial()` to keep the threshold values in **one place** in the code — same pattern as W11's `ATOMIC_THRESHOLD`.

In [ ]:
COMFORT_LOW, COMFORT_HIGH = 18.0, 28.0    # comfort-critical band
PRICE_HIGH = 18.0                         # cents/kWh threshold for save_energy

desire_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You pick one goal for a smart home given its current beliefs.\n"
     "Rules:\n"
     "- If indoor_temp is outside the {comfort_low}-{comfort_high} C band, "
     "goal = 'comfort', priority = 1.\n"
     "- Else if energy_price_now > {price_high} cents/kWh, "
     "goal = 'save_energy', priority = 2.\n"
     "- Otherwise goal = 'comfort', priority = 3.\n"
     'Return ONLY a JSON object: {{"goal": "comfort"|"save_energy"|"safety", '
     '"priority": <int 1-3>}}'),
    ("human", "Beliefs: {belief}")
]).partial(
    comfort_low=str(COMFORT_LOW),
    comfort_high=str(COMFORT_HIGH),
    price_high=str(PRICE_HIGH),
)

desire_chain = desire_prompt | llm | JsonOutputParser()
print(f"desire_chain ready (thresholds: comfort {COMFORT_LOW}-{COMFORT_HIGH}C, price > {PRICE_HIGH}c).")

### 2.3 Intention prompt — form an action and a reason

In `02_patterns` the intention was picked from a small enumeration with hand-written rationales. Here the LLM produces both the action and the reason.

In [ ]:
intention_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You decide the smart home's next action given its beliefs and chosen desire.\n"
     "Available actions: HEAT_ON, AC_ON, STANDBY, DEFER_FLEXIBLE_LOADS.\n"
     "Pick the single most appropriate one for this combination of belief and desire.\n"
     'Return ONLY a JSON object: {{"action": <one of the available actions>, '
     '"reason": <one short sentence citing the relevant numbers>}}'),
    ("human", "Beliefs: {belief}\nDesire: {desire}")
])
intention_chain = intention_prompt | llm | JsonOutputParser()
print("intention_chain ready.")

## Section 3 — Three nodes that each call an LLM

These functions are the production replacement for the three nodes in `02_patterns`. The signatures match. The graph wiring will match too. Only the *body* of each node changes: instead of an if/elif, it calls one LLM.

Each node also has a **try/except fallback to a rule-based default**, so if the LLM call fails (network blip, malformed JSON), the agent still produces something sensible. This is a small but important production-grade habit.

### 3.1 The graph state — with a reducer for the audit log

We carry the typed Belief / Desire / Intention as **serialized dicts** inside the TypedDict — same TypedDict-of-dicts pattern as the bridge notebook. Each node hydrates to the Pydantic model, does its work, and dehydrates back.

**One critical detail:** `decision_log` uses an `Annotated[list[dict], add]` reducer. Without this, LangGraph's default behavior is **overwrite-per-key**: when a node returns `{"decision_log": [new_entry]}`, the new list REPLACES the old one. So the audit trail would show only the most recent entry instead of accumulating.

With `Annotated[list, add]`, LangGraph uses `operator.add` to **concatenate** the lists. The node emits *just the new entry as a list of one*, and LangGraph appends it to the existing log. This is exactly how LangGraph's built-in `messages` field works (with `add_messages` reducer).

In [ ]:
class BDIState(TypedDict):
    # Input
    observation: str
    # BDI reasoning slots (filled in by the three nodes, in order)
    belief: Optional[dict]      # Belief as dict (default reducer: overwrite)
    desire: Optional[dict]      # overwrite per cycle
    intention: Optional[dict]   # overwrite per cycle
    # Production audit trail — append-only via the `add` reducer
    decision_log: Annotated[list[dict], add]

print("BDIState schema declared (decision_log uses an `add` reducer).")

### 3.2 `update_belief` node — LLM extracts Belief from observation

In [ ]:
def update_belief(state: BDIState) -> BDIState:
    try:
        b_dict = belief_chain.invoke({"observation": state["observation"]})
        belief = Belief(**b_dict)
    except Exception as e:
        print(f"  [belief LLM failed: {type(e).__name__}, using default Belief]")
        belief = Belief()  # sensible default
    print(f"[BELIEF]    {belief}")
    return {**state, "belief": belief.model_dump()}

print("update_belief defined.")

### 3.3 `pick_desire` node — LLM picks goal given Belief

In [ ]:
def pick_desire(state: BDIState) -> BDIState:
    belief_json = json.dumps(state["belief"])  # the LLM sees the belief as JSON
    try:
        d_dict = desire_chain.invoke({"belief": belief_json})
        desire = Desire(**d_dict)
    except Exception as e:
        print(f"  [desire LLM failed: {type(e).__name__}, falling back to rule-based]")
        b = Belief(**state["belief"])
        if b.indoor_temp < COMFORT_LOW or b.indoor_temp > COMFORT_HIGH:
            desire = Desire(goal="comfort", priority=1)
        elif b.energy_price_now > PRICE_HIGH:
            desire = Desire(goal="save_energy", priority=2)
        else:
            desire = Desire(goal="comfort", priority=3)
    print(f"[DESIRE]    {desire}")
    return {**state, "desire": desire.model_dump()}

print("pick_desire defined.")

### 3.4 `form_intention` node — LLM forms action+reason

This node returns the new intention **as a single-element list** for `decision_log`. The `add` reducer on the state schema takes care of concatenation — we don't read-and-rebuild the log here. That's the whole point of the reducer: append semantics live in the schema, not in every caller.

In [ ]:
def form_intention(state: BDIState) -> BDIState:
    belief_json = json.dumps(state["belief"])
    desire_json = json.dumps(state["desire"])
    try:
        i_dict = intention_chain.invoke({"belief": belief_json, "desire": desire_json})
        intention = Intention(**i_dict)
    except Exception as e:
        print(f"  [intention LLM failed: {type(e).__name__}, falling back to rule-based]")
        b = Belief(**state["belief"])
        d = Desire(**state["desire"])
        if d.goal == "comfort":
            if b.indoor_temp < 20:
                intention = Intention(action="HEAT_ON", reason=f"indoor {b.indoor_temp}C below 20")
            elif b.indoor_temp > 26:
                intention = Intention(action="AC_ON", reason=f"indoor {b.indoor_temp}C above 26")
            else:
                intention = Intention(action="STANDBY", reason="within comfort band")
        elif d.goal == "save_energy":
            intention = Intention(action="DEFER_FLEXIBLE_LOADS",
                                  reason=f"price {b.energy_price_now}c is high")
        else:
            intention = Intention(action="STANDBY", reason="fallback")
    print(f"[INTENTION] {intention}")

    intention_dict = intention.model_dump()
    return {
        **state,
        "intention": intention_dict,
        # Emit a list of ONE — the `add` reducer concatenates it onto the existing log
        "decision_log": [intention_dict],
    }

print("form_intention defined.")

## Section 4 — Build the graph (same topology as `02_patterns`)

If you put this graph side-by-side with the one in `02_patterns`, you'll see they're identical: same nodes, same edges, same START/END. Only the *body* of each node is different.

In [ ]:
bdi_graph = StateGraph(BDIState)
bdi_graph.add_node("update_belief", update_belief)
bdi_graph.add_node("pick_desire", pick_desire)
bdi_graph.add_node("form_intention", form_intention)

bdi_graph.add_edge(START, "update_belief")
bdi_graph.add_edge("update_belief", "pick_desire")
bdi_graph.add_edge("pick_desire", "form_intention")
bdi_graph.add_edge("form_intention", END)

def render(graph):
    try:
        display(Image(graph.get_graph().draw_mermaid_png()))
    except Exception as e:
        print(f"PNG rendering failed ({type(e).__name__}). ASCII fallback:")
        print(graph.get_graph().draw_ascii())

# Preview without checkpointer (we'll compile WITH one in Section 5)
preview = bdi_graph.compile()
render(preview)

## Section 5 — Add the checkpointer for persistence

### 5.1 Why a checkpointer

Without one, the agent forgets everything between calls. With one, the `decision_log` accumulates across invocations — so a `/audit` endpoint can later return *every* intention this home has formed.

### 5.2 thread_id keys the cabinet

`thread_id` is the home's account number. Every invocation passes one. The checkpointer stores per-thread state.

In [ ]:
checkpointer = MemorySaver()
agent = bdi_graph.compile(checkpointer=checkpointer)

# Each home gets its own thread_id
home_thread = {"configurable": {"thread_id": "home_42"}}
print("Agent compiled with checkpointer (thread_id=home_42).")

## Section 6 — Run several BDI cycles and watch reasoning accumulate

Three different observations for the same home. The LLM extracts belief, picks desire, forms intention each time. Watch the `decision_log` grow across cycles.

**Important:** we pass ONLY `{"observation": obs}` each iteration. We do NOT pass `decision_log: []` — that would (with a non-reducer field) overwrite whatever the checkpointer holds. With our `Annotated[list, add]` reducer the loop would still work, but the cleaner habit is to pass only the fields that genuinely change each call. The checkpointer + reducers handle the rest.

In [ ]:
observations = [
    "Cold winter morning at 8 AM, indoor temp is 17.5C, outdoor 5C, energy price 12 cents/kWh, family is home.",
    "Hot afternoon at 2 PM, indoor temp is 23C, outdoor 32C, energy price 22 cents/kWh, family is home.",
    "Mild evening at 7 PM, indoor temp is 29C, outdoor 27C, energy price 20 cents/kWh, family just got home.",
]

for i, obs in enumerate(observations, 1):
    print(f"\n=== Cycle {i}: {obs[:70]}... ===")
    result = agent.invoke(
        {"observation": obs},   # ONLY the new observation; checkpointer holds the rest
        config=home_thread,
    )

print(f"\nFinal decision_log length: {len(result['decision_log'])}  (expected: 3)")

**What just happened:**

- Three observations → three full BDI cycles.
- Each cycle: belief extracted from text by an LLM → desire picked by an LLM → intention formed by an LLM.
- `decision_log` accumulated all three intentions because of the `add` reducer on the schema. Each `form_intention` call returned `[one_new_intention]`; LangGraph concatenated it onto the existing log.
- The other slots (`belief`, `desire`, `intention`) **overwrite** each cycle — they're snapshots of the most recent reasoning, not history.

The shape is identical to `02_patterns` — three nodes in a chain. What's different is that the *substance* of each node is now LLM-driven, the agent remembers across invocations, and the audit log is **append-only by construction**, not by convention.

## Section 7 — Observability: read the audit trail

`agent.get_state(...)` returns the latest persisted state for the thread without running another cycle. This is exactly how a production `/audit` endpoint would work.

In [ ]:
state = agent.get_state(home_thread)
home_state = state.values

print(f"Most recent reasoning:")
print(f"  Belief    : {home_state['belief']}")
print(f"  Desire    : {home_state['desire']}")
print(f"  Intention : {home_state['intention']}")
print()
print(f"Decision log ({len(home_state['decision_log'])} entries):")
for i, intent in enumerate(home_state["decision_log"], 1):
    print(f"  {i}. action={intent['action']} reason={intent['reason']}")

**The production payoff:** any consumer (mobile app, audit dashboard, regulator) can read these three slots and answer *"what does this home believe, what is it trying to achieve, and what did it most recently decide?"* The `decision_log` answers *"and what has it decided over the past N cycles?"*

This is what audit-trail-grade explainability looks like. It's the same payoff `02_patterns` promised conceptually — now backed by persistent storage.

## Section 8 — What's still missing for true production

This notebook stops short of:

1. **Cost engineering on the three sub-chains.** Belief extraction is structural — a cheap small model is fine. Intention formation is reasoning-heavy — use a bigger model. Today we use one model throughout for simplicity.
2. **Anthropic prompt caching (W13).** The three system prompts are stable across calls. Caching them slashes per-call cost by ~70-80%.
3. **`with_retry` per sub-chain.** Today we have try/except with rule-based fallback. Production also adds exponential-backoff retries before falling back.
4. **LangSmith integration (W17).** Each of the three nodes naturally becomes a named child run in LangSmith, so you can see latency and cost per BDI step.
5. **Production checkpointer.** `MemorySaver` loses everything on process restart. Use `SqliteSaver` for single-process apps or `PostgresSaver` for multi-process production.
6. **Multi-home isolation at the DB layer.** `thread_id` isolates state in memory; production needs row-level security in the database checkpointer.
7. **Drift monitoring (W16).** Over weeks, the LLM's belief-extraction accuracy will drift as observation phrasing evolves. Evidently catches this.
8. **Real device adapters.** Today `Intention.action` is a string. In production it routes to a Z-Wave / Zigbee / Matter driver.

The pattern itself — three BDI nodes with LLM bodies, Pydantic state, checkpointer, audit log — is the production-ready spine. Everything above is operational concerns layered on top.